In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
data = pd.read_csv('data_reviews_purchase.csv', usecols=['user_id', 'product_id', 'rating'], encoding='latin-1')
data

,user_id,product_id,rating
0,0,0,4
1,1,0,5
2,2,0,5
3,3,0,5
4,4,0,5
...,...,...,...
369094,115066,2235,5
369095,304704,2235,5
369096,304705,2235,5
369097,304706,2235,5


In [3]:
print(data.duplicated().sum())
data.drop_duplicates(inplace=True)
data

30065


,user_id,product_id,rating
0,0,0,4
1,1,0,5
2,2,0,5
3,3,0,5
4,4,0,5
...,...,...,...
369094,115066,2235,5
369095,304704,2235,5
369096,304705,2235,5
369097,304706,2235,5


In [4]:
data = data.pivot_table(index='user_id', columns='product_id', values='rating', aggfunc='mean').astype('float32')
data

product_id,0,1,2,3,4,5,6,7,8,9,...,2226,2227,2228,2229,2230,2231,2232,2233,2234,2235
user_id,,,,,,,,,,,,,,,,,,,,,
0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
304703,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0
304704,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0
304705,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0


In [5]:
nan_ratio = data.isna().sum().sum() / (data.shape[0] * data.shape[1])
print(f"Tỷ lệ NaN của data: {nan_ratio:.4f}")

Tỷ lệ NaN của data: 0.9995


In [7]:
class MatrixFactorizationRecommender():

  def __init__(self, data: pd.DataFrame, k: int, alpha: float, lam: float, tol: float = 1e-4 ,max_iters: int = 100):
    """
    @params
      - data: dataframe với dòng~user, cột~item, value ~ rating
      - K: số chiều của nhân tố ẩn, cho cả U và V
      - alpha : cỡ bước
      - lam : tham số hiệu chỉnh
      - max_iters: số vòng lặp
    """
    self.data = data
    self.R = self.data.to_numpy().astype(np.float32)
    self.m, self.n = self.data.shape
    self.k = k
    self.tol = tol
    self.max_iters = max_iters
    # Khởi tạo các ma trận nhân tố ẩn U(m x k) và V(n x k)
    self.U = np.random.randn(self.m, k).astype(np.float32)
    self.V = np.random.randn(self.n, k).astype(np.float32)

    # Khởi tạo các siêu tham số (hyperparameter)
    self.alpha = alpha
    self.lam = lam

  def init_biases(self):
    """
    Khởi tạo các bias cho người dùng và item
    """
    self.b_user = np.zeros(self.m).astype(np.float32) # bias đối với users
    self.b_item = np.zeros(self.n).astype(np.float32) # bias đối với item

  def sgd(self):
    """
    Hàm Stochastic Gradient Descent để cập nhật bias và ma trận nhân tố ẩn U, V
    """
    for i, j, r in self.S:
      # Ước lượng rating
      prediction = self.get_rating(i, j)
      # Tính sai số
      e = r - prediction

      # Cập nhật biases
      self.b_user[i] += self.alpha * (e - self.lam * self.b_user[i])
      self.b_item[j] += self.alpha * (e - self.lam * self.b_item[j])

      # Tạo ma trận U, V trung gian
      U_i = self.U[i, :]
      V_i = self.V[j, :]

      # Cập nhật ma trận U và V
      self.U[i, :] += self.alpha * (e * V_i - self.lam * U_i)
      self.V[j, :] += self.alpha * (e * U_i - self.lam * V_i)

  def get_rating(self, i: int, j: int) -> float:
    """
    Dự đoán rating của user i đối với item j
    """
    pred = self.b_user[i] + self.b_item[j] + self.U[i, :] @ self.V[j, :].T
    return pred

  def full_matrix(self) -> pd.DataFrame:
    """
      Cập nhật ma trận đầy đủ sử dụng các bias, ma trận nhân tố ẩn U và V
    """
    predicted_R = self.b_user[:, None] + self.b_item[None, :] + self.U @ self.V.T
    return pd.DataFrame(np.clip(predicted_R, 1, 5),
                        index=self.data.index, columns=self.data.columns)

  def rmse(self) -> float:
    """
      Hàm tính căn bậc hai trung bình bình phương sai số (RMSE)
    """
    # Ma trận rating dự đoán
    predicted = self.full_matrix().to_numpy()
    # Lấy các vị trí mà r_ij != nan
    mask = ~np.isnan(self.R)
    return np.sqrt(np.mean((self.R[mask] - predicted[mask])**2))

  def loss(self) -> float:
    return 0.5 * (self.rmse() ** 2 + self.lam * (
        np.sum(self.U**2) + np.sum(self.V**2) +
        np.sum(self.b_user**2) + np.sum(self.b_item**2)
    ))

  def train(self):
    """
    Huấn luyện mô hình bằng phương pháp Stochastic Gradient Descent (SGD)
    """
    self.init_biases()
    # Tạo tập S = [(i,j,r_ij): r_ij != nan]
    self.S = [(i, j, self.R[i, j])
              for i in range(self.m)
              for j in range(self.n)
              if not np.isnan(self.R[i, j])]

    training_process = [] # Danh sách lưu quá trình huấn luyện
    patience = 5  # Số lần lặp không cải thiện trước khi dừng
    best_rmse = float('inf') # Biến lưu giá trị RMSE tốt nhất
    no_improve = 0 # Biến đếm số lần không cải thiện

    for iter in range(self.max_iters):
      np.random.shuffle(self.S)
      # Cập nhật U và V
      self.sgd()
      # Tính sai số rmse
      error = self.rmse()
      # Hàm loss
      loss = self.loss()
      # Lưu lần lặp và sai số tương ứng
      training_process.append((iter, loss, error))
      # In thông tin quá trình huấn luyện
      if (iter + 1) % 2 == 0:
        print(f"Iter {iter+1}: Loss = {loss:.8f}, RMSE = {error:.8f}")
      # Kiểm tra điều kiện dừng sớm
      if error < best_rmse - self.tol:
          best_rmse = error
          no_improve = 0
      else:
          no_improve += 1
          if no_improve >= patience:
              print(f"Early stopping at iteration {iter+1}")
              break

    return training_process

  def pred_for_users(self, target_user: int) -> pd.DataFrame:
    rated = self.data.loc[target_user].dropna().index
    unrated = self.data.columns.difference(rated)
    return (self.full_matrix().loc[[target_user], unrated]
            .T.rename(columns={target_user: 'pred_rating'})
            .sort_values('pred_rating', ascending=False)
            .round(2))


In [5]:
from sklearn.model_selection import train_test_split
data_train, data_test = train_test_split(data, train_size=0.25, random_state=42)
data_train, data_test = train_test_split(data_train, train_size=0.25, random_state=42)

In [7]:
data_train

product_id,0,1,2,3,4,5,6,7,8,9,...,2226,2227,2228,2229,2230,2231,2232,2233,2234,2235
user_id,,,,,,,,,,,,,,,,,,,,,
188151,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
74063,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
251537,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
119682,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
104319,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199133,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
297180,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
124433,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
from itertools import product

def tune_hyperparameters(data: pd.DataFrame):
    best_config = None
    best_rmse = float('inf')
    results = []

    k_list = [10, 20, 30]
    alpha_list = [0.001, 0.005, 0.01]
    lam_list = [0.01, 0.05, 0.1]

    for k, alpha, lam in product(k_list, alpha_list, lam_list):
        print(f"Training with k={k}, alpha={alpha}, lam={lam}")
        model = MatrixFactorizationRecommender(data, k=k, alpha=alpha, lam=lam, tol=1e-4, max_iters=100)
        training_process = model.train()
        final_rmse = training_process[-1][2]  # Lấy RMSE cuối cùng
        results.append((k, alpha, lam, final_rmse))
        print(f"Final RMSE: {final_rmse:.4f}")
        if final_rmse < best_rmse:
            best_rmse = final_rmse
            best_config = (k, alpha, lam)

    print(f"\n✅ Best config: k={best_config[0]}, alpha={best_config[1]}, lam={best_config[2]} -> RMSE: {best_rmse:.4f}")
    return best_config, results

In [9]:
tune_hyperparameters(data_train)

Training with k=10, alpha=0.001, lam=0.01
Iter 2: Loss = 1064.28845215, RMSE = 3.28152704
Iter 4: Loss = 1060.53942871, RMSE = 3.03539824
Iter 6: Loss = 1058.34228516, RMSE = 2.77223396
Iter 8: Loss = 1057.11486816, RMSE = 2.54526258
Iter 10: Loss = 1056.50695801, RMSE = 2.36484098
Iter 12: Loss = 1056.28991699, RMSE = 2.22088861
Iter 14: Loss = 1056.32568359, RMSE = 2.10230303
Iter 16: Loss = 1056.53491211, RMSE = 2.00242400
Iter 18: Loss = 1056.86486816, RMSE = 1.91582108
Iter 20: Loss = 1057.27929688, RMSE = 1.83793950
Iter 22: Loss = 1057.75646973, RMSE = 1.76657927
Iter 24: Loss = 1058.28002930, RMSE = 1.70043862
Iter 26: Loss = 1058.83642578, RMSE = 1.63809335
Iter 28: Loss = 1059.41796875, RMSE = 1.57940793
Iter 30: Loss = 1060.01647949, RMSE = 1.52354193
Iter 32: Loss = 1060.62561035, RMSE = 1.47016573
Iter 34: Loss = 1061.24084473, RMSE = 1.41887021
Iter 36: Loss = 1061.85766602, RMSE = 1.36927021
Iter 38: Loss = 1062.47363281, RMSE = 1.32158887
Iter 40: Loss = 1063.08569336, 

((30, 0.01, 0.01),
 [(10, 0.001, 0.01, np.float32(0.52725804)),
  (10, 0.001, 0.05, np.float32(0.5747831)),
  (10, 0.001, 0.1, np.float32(0.6805753)),
  (10, 0.005, 0.01, np.float32(0.19189928)),
  (10, 0.005, 0.05, np.float32(0.27502203)),
  (10, 0.005, 0.1, np.float32(0.4105528)),
  (10, 0.01, 0.01, np.float32(0.12229317)),
  (10, 0.01, 0.05, np.float32(0.22113644)),
  (10, 0.01, 0.1, np.float32(0.35466772)),
  (20, 0.001, 0.01, np.float32(0.33482918)),
  (20, 0.001, 0.05, np.float32(0.40191057)),
  (20, 0.001, 0.1, np.float32(0.5189158)),
  (20, 0.005, 0.01, np.float32(0.16364053)),
  (20, 0.005, 0.05, np.float32(0.2509934)),
  (20, 0.005, 0.1, np.float32(0.38032088)),
  (20, 0.01, 0.01, np.float32(0.10306991)),
  (20, 0.01, 0.05, np.float32(0.20233598)),
  (20, 0.01, 0.1, np.float32(0.31888333)),
  (30, 0.001, 0.01, np.float32(0.2673119)),
  (30, 0.001, 0.05, np.float32(0.34617862)),
  (30, 0.001, 0.1, np.float32(0.46764752)),
  (30, 0.005, 0.01, np.float32(0.14496388)),
  (30, 0.0

In [8]:
mf = MatrixFactorizationRecommender(data_train, k=30, alpha=0.01, lam=0.01, tol=1e-4, max_iters=300)
training_process = mf.train()

Iter 2: Loss = 3147.34162526, RMSE = 1.07127285
Iter 4: Loss = 3147.57873779, RMSE = 0.47497430
Iter 6: Loss = 3146.66609840, RMSE = 0.32356885
Iter 8: Loss = 3145.24144944, RMSE = 0.27115288
Iter 10: Loss = 3143.68757789, RMSE = 0.24653760
Iter 12: Loss = 3142.10050915, RMSE = 0.23268715
Iter 14: Loss = 3140.51342803, RMSE = 0.22216225
Iter 16: Loss = 3138.92589334, RMSE = 0.21485269
Iter 18: Loss = 3137.34639926, RMSE = 0.20838313
Iter 20: Loss = 3135.78999468, RMSE = 0.20153004
Iter 22: Loss = 3134.23468680, RMSE = 0.19684662
Iter 24: Loss = 3132.68541897, RMSE = 0.19095270
Iter 26: Loss = 3131.14974558, RMSE = 0.18571797
Iter 28: Loss = 3129.61707471, RMSE = 0.18138197
Iter 30: Loss = 3128.08719697, RMSE = 0.17683023
Iter 32: Loss = 3126.55917497, RMSE = 0.17385322
Iter 34: Loss = 3125.05233809, RMSE = 0.16860063
Iter 36: Loss = 3123.54296372, RMSE = 0.16485581
Iter 38: Loss = 3122.03866572, RMSE = 0.16149749
Iter 40: Loss = 3120.53988944, RMSE = 0.15741311
Iter 42: Loss = 3119.050

In [9]:
mf.full_matrix()

product_id,0,1,2,3,4,5,6,7,8,9,...,2226,2227,2228,2229,2230,2231,2232,2233,2234,2235
user_id,,,,,,,,,,,,,,,,,,,,,
188151,4.013648,5.000000,5.000000,5.000000,5.000000,2.595840,1.000000,2.833662,1.000000,1.000000,...,1.000000,1.0,1.000000,1.000000,4.844554,5.000000,4.248768,1.000000,1.000000,1.000000
74063,3.220566,4.737885,1.000000,1.000000,5.000000,5.000000,5.000000,1.000000,5.000000,5.000000,...,1.645062,1.0,2.322158,1.175840,1.000000,1.000000,5.000000,5.000000,2.267978,2.556157
251537,4.211997,4.993951,4.166982,1.000000,5.000000,5.000000,1.000000,2.100649,5.000000,1.000000,...,1.000000,5.0,1.000000,1.000000,2.352625,1.000000,1.000000,1.000000,5.000000,5.000000
119682,4.410141,4.209439,2.735364,1.000000,5.000000,1.441896,5.000000,3.194234,1.000000,5.000000,...,1.000000,5.0,1.000000,5.000000,1.000000,3.664624,5.000000,3.036735,3.613606,5.000000
104319,5.000000,4.243117,1.000000,1.000000,4.299594,3.749542,5.000000,1.000000,1.000000,1.000000,...,1.436350,5.0,1.000000,5.000000,5.000000,1.000000,4.609996,1.000000,1.000000,3.137614
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199133,4.992494,5.000000,1.799427,1.000000,3.864549,5.000000,3.707305,1.000000,3.799060,4.132865,...,1.000000,1.0,1.000000,1.000000,1.159033,2.144337,5.000000,1.189018,1.000000,5.000000
297180,3.110414,4.322889,1.000000,1.000000,3.714491,1.000000,5.000000,2.395211,1.000000,1.000000,...,5.000000,5.0,4.040119,4.005052,1.000000,1.000000,1.000000,1.302445,1.000000,4.033970
124433,4.422815,5.000000,1.000000,2.549551,4.771904,1.000000,1.000000,1.000000,4.941782,1.000000,...,1.000000,1.0,5.000000,1.000000,1.240318,1.000000,1.000000,1.000000,5.000000,1.000000


In [10]:
target_user = 104319
print(f"Dự đoán rating cho các item mà user {target_user} chưa đánh giá: \n")
(mf.pred_for_users(target_user))

Dự đoán rating cho các item mà user 104319 chưa đánh giá: 



user_id,pred_rating
product_id,
0,5.0
1608,5.0
1568,5.0
1066,5.0
388,5.0
...,...
532,1.0
1326,1.0
1324,1.0
